In [ ]:
import duckdb
import pandas as pd
from datetime import date, timedelta
SOURCE_DB = "adworks.duckdb"
DW_DB = "warehouse.duckdb"

def build_dim_date(con):
    """
    Tự generate DimDate từ 2000-01-01 đến 2020-12-31
    """
    start = date(2000, 1, 1)
    end = date(2020, 12, 31)

    rows = []
    cur = start
    while cur <= end:
        rows.append(
            {
                "DateKey": int(cur.strftime("%Y%m%d")),
                "FullDate": cur,
                "Day": cur.day,
                "Month": cur.month,
                "MonthName": cur.strftime("%B"),
                "Quarter": (cur.month - 1) // 3 + 1,
                "Year": cur.year,
                "Week": int(cur.strftime("%W")),
            }
        )
        cur += timedelta(days=1)

    dim_date_df = pd.DataFrame(rows)

    con.execute("CREATE SCHEMA IF NOT EXISTS dwh;")
    con.execute("DROP TABLE IF EXISTS dwh.DimDate;")
    con.register("dim_date_df", dim_date_df)
    con.execute("CREATE TABLE dwh.DimDate AS SELECT * FROM dim_date_df;")
    con.unregister("dim_date_df")


def build_dim_product(con):
    """
    DimProduct lấy từ src.Production.Product + ProductSubcategory + ProductCategory
    """
    con.execute("CREATE SCHEMA IF NOT EXISTS dwh;")
    con.execute("DROP TABLE IF EXISTS dwh.DimProduct;")

    con.execute(
        """
        CREATE TABLE dwh.DimProduct AS
        SELECT
            ROW_NUMBER() OVER ()                AS ProductKey,
            p.ProductID,
            p.Name                              AS ProductName,
            p.ProductNumber,
            p.MakeFlag,
            p.StandardCost,
            p.ListPrice,
            sc.Name                             AS ProductSubcategory,
            c.Name                              AS ProductCategory,
            p.SafetyStockLevel,
            p.ReorderPoint
        FROM src.Production.Product p
        LEFT JOIN src.Production.ProductSubcategory sc
          ON p.ProductSubcategoryID = sc.ProductSubcategoryID
        LEFT JOIN src.Production.ProductCategory c
          ON sc.ProductCategoryID = c.ProductCategoryID;
        """
    )

def build_dim_vendor(con):
    """
    DimVendor lấy từ src.Purchasing.Vendor + src.Purchasing.ProductVendor
    """
    con.execute("CREATE SCHEMA IF NOT EXISTS dwh;")
    con.execute("DROP TABLE IF EXISTS dwh.DimVendor;")

    con.execute(
        """
        CREATE TABLE dwh.DimVendor AS
        SELECT DISTINCT
            ROW_NUMBER() OVER ()                  AS VendorKey,
            v.BusinessEntityID                    AS VendorID,
            v.Name                                AS VendorName,
            v.CreditRating,
            v.PreferredVendorStatus,
            v.ActiveFlag,
            pv.AverageLeadTime,
            pv.StandardPrice
        FROM src.Purchasing.Vendor v
        LEFT JOIN src.Purchasing.ProductVendor pv
          ON v.BusinessEntityID = pv.BusinessEntityID;
        """
    )

def build_dim_location(con):
    """
    DimLocation từ src.Production.Location
    """
    con.execute("CREATE SCHEMA IF NOT EXISTS dwh;")
    con.execute("DROP TABLE IF EXISTS dwh.DimLocation;")

    con.execute(
        """
        CREATE TABLE dwh.DimLocation AS
        SELECT
            ROW_NUMBER() OVER ()          AS LocationKey,
            l.LocationID,
            l.Name                        AS LocationName,
            l.CostRate,
            l.Availability
        FROM src.Production.Location l;
        """
    )

def build_fact_purchasing(con):
    """
    FactPurchasing: grain = 1 dòng / product / purchase order detail
    Map sang DimProduct, DimVendor, DimDate (OrderDate + DueDate)
    """
    con.execute("CREATE SCHEMA IF NOT EXISTS dwh;")
    con.execute("DROP TABLE IF EXISTS dwh.FactPurchasing;")

    con.execute(
        """
        CREATE TABLE dwh.FactPurchasing AS
        SELECT
            ROW_NUMBER() OVER ()                                AS PurchaseKey,
            d.PurchaseOrderID,
            dp.ProductKey,
            dv.VendorKey,
            ddOrder.DateKey                                     AS DateKey,
            ddDue.DateKey                                       AS DueDateKey,
            d.OrderQty,
            d.UnitPrice,
            COALESCE(d.LineTotal, d.OrderQty * d.UnitPrice)     AS LineTotal,
            h.Status,
            h.Freight,
            d.ReceivedQty,
            d.RejectedQty
        FROM src.Purchasing.PurchaseOrderDetail d
        JOIN src.Purchasing.PurchaseOrderHeader h
          ON d.PurchaseOrderID = h.PurchaseOrderID
        JOIN dwh.DimProduct dp
          ON d.ProductID = dp.ProductID
        JOIN dwh.DimVendor dv
          ON h.VendorID = dv.VendorID
        JOIN dwh.DimDate ddOrder
          ON CAST(h.OrderDate AS DATE) = ddOrder.FullDate
        JOIN dwh.DimDate ddDue
          ON CAST(d.DueDate  AS DATE) = ddDue.FullDate;
        """
    )

def build_fact_inventory(con):
    """
    FactInventory: snapshot theo Product-Location-Bin-Shelf-Date
    Dùng src.Production.ProductInventory + DimProduct + DimLocation + DimDate
    """
    con.execute("CREATE SCHEMA IF NOT EXISTS dwh;")
    con.execute("DROP TABLE IF EXISTS dwh.FactInventory;")

    con.execute(
        """
        CREATE TABLE dwh.FactInventory AS
        SELECT
            ROW_NUMBER() OVER ()         AS InventoryKey,
            dp.ProductKey,
            dl.LocationKey,
            dd.DateKey,
            pi.Quantity                  AS QuantityOnHand,
            pi.Shelf,
            pi.Bin
        FROM src.Production.ProductInventory pi
        JOIN dwh.DimProduct dp
          ON pi.ProductID = dp.ProductID
        JOIN dwh.DimLocation dl
          ON pi.LocationID = dl.LocationID
        JOIN dwh.DimDate dd
          ON CAST(pi.ModifiedDate AS DATE) = dd.FullDate;
        """
    )

In [6]:
con = duckdb.connect(DW_DB)
con.execute(f"ATTACH '{SOURCE_DB}' AS src (READ_ONLY);")

build_dim_date(con)
build_dim_product(con)
build_dim_vendor(con)
build_dim_location(con)
build_fact_purchasing(con)
build_fact_inventory(con)
con.close()